In [5]:
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import pandas as pd
import time
import random

In [2]:
session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,/;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
}

session.headers.update(headers)
# [hyd,blr,chennai,coim,vishaka,delhi,mumbai,ahmadabad,kolkata,pune,jaipur,lucknow,nagpur,Surat,patna,Agra,Vishakapatnam,Varanasi,Mysore,Guntur,Kerala,Ranchi,Guwahati,Mirzapur,vadodara,
# vijaywada,nashik,kollam,raipur,madurai,meerut,srinagar,aurangabad,jodhpur,kota,jabalpur,gwalior,allahabad,amritsar,dhanbad,aligarh,moradabad,chandigarh,]

In [ ]:
links=["https://www.99acres.com/search/property/buy/residential-all/vadodara?city=96&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/buy/residential-all/vijayawada?city=61&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/buy/residential-all/nashik?city=151&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/buy/residential-all/madurai?city=188&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/buy/residential-all/meerut?city=207&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N",
       "https://www.99acres.com/search/property/buy/residential-all/jodhpur?city=179&bedroom_num=1%2C2%2C3%2C4%2C4%2C4&class=O%2CB%2CA&property_type=1%2C4%2C2%2C90&preference=S&area_unit=1&budget_min=0&res_com=R&isPreLeased=N"
       
      ]


Names=[]
types=[]
price=[]
Area=[]
city=[]
status=[]
date=[]
builder=[]
for i in links:
    url=i
    page=None
    page=requests.get(url,headers=headers)
    print(page)
    for attempt in range(3):
        
        try:
            page = session.get(url, headers=headers, timeout=10)
            
            if page.status_code == 200:
                print(f"200 for {url}")
                soup = BeautifulSoup(page.text, "html.parser")
                break
            
            else:
                print(f"Attempt {attempt+1} failed:", page.status_code)
                time.sleep(random.uniform(5,10))
        
        except requests.exceptions.RequestException as e:
            print("Connection Error:", e)
            time.sleep(random.uniform(5,10))
    
    else:
        print("Skipping:", url) 
    for rating in soup.find_all("div", class_="PseudoTupleRevamp__locRatings"):
        rating.decompose()
    for new in soup.find_all("div",class_="PseudoTupleRevamp__ribbon"):
        new.decompose()
    Nam=soup.find_all("div",class_="PseudoTupleRevamp__projectHeadContainer")
    for i in Nam:
        text = i.text
        
        match = re.search(r'(\d+(?:,\s*\d+)*)\s*BHK', text)
        
        if match:
            nums = re.findall(r'\d+', match.group(1))
            
            for n in nums:
                Names.append(re.findall(r"(.+?)\d+", text)[0])
    for i in Nam:
        typ=re.findall(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        if typ:
            nums_1 = re.findall(r'\d+', typ[0])
            types.extend([int(n[-1]) for n in nums_1]) 
    pri=soup.find_all("div",class_="configs__configCards")
    for i in pri:
        if re.search(r"[5-9]\s*BHK|\d{2,}\s*BHK", i.text):
            data = re.search(r"^.*?(?=(?:[5-9]|1\d|20)\s*BHK)", i.text)
            a=(re.findall(r"₹[\d\.]+(?:\s*(?:Cr|L))?(?:\s*-\s*[\d\.]+(?:\s*(?:Cr|L))?)?",data.group()))
            if a:
                price.extend(a)
            else:
                price.extend("NA")
        
        elif "Land" in i.text:
            price.extend(re.findall(r"₹[\d\.]+(?:\s*(?:Cr|L))?(?:\s*-\s*[\d\.]+(?:\s*(?:Cr|L))?)?",i.text.split("Land")[0]))
        
        elif "Price on Request" in i.text:
            result=(re.findall(r'Price on Request|₹[\d\.]+(?:\s*(?:Cr|L))?(?:\s*-\s*[\d\.]+(?:\s*(?:Cr|L))?)?',i.text))
            price.extend(['Price on Request' if r=='' else r for r in result])
        else:
            price.extend(re.findall(r"₹[\d\.]+(?:\s*(?:Cr|L))?(?:\s*-\s*[\d\.]+(?:\s*(?:Cr|L))?)?", i.text))
    ar=soup.find_all("h2",class_="PseudoTupleRevamp__subHeading ellipsis PseudoTupleRevamp__projectHeading")
    for i in ar:
        p=re.search(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        nums=re.findall(r'\d+', p.group(1))
        if p:
            for n in nums:
                arr=re.findall("in\s*(.+?),\s\w+",i.text)
                if len(arr)>0:
                    Area.append(arr[0])
                else:
                    Area.append("NA")
    for i in ar:
        p=re.search(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        nums=re.findall(r'\d+', p.group(1))
        if p:
            for n in nums:
                city.append(re.findall("\w+$",i.text)[0])
    bui=soup.find_all("div",class_="PseudoTupleRevamp__tupleWrapProject undefined")
    for i in bui:
        p=re.search(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        nums=re.findall(r'\d+', p.group(1))
        if p:
            for n in nums:
                builder.append(i.text.split("Builder")[1].split("Brochure")[0])
    for i in bui:
        p=re.search(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        nums=re.findall(r'\d+', p.group(1))
        if p:
            for n in nums:
                status.append(re.findall(r"\w+\s\w+\s?\w*.?",i.text)[0].strip())
    for i in bui:
        p=re.search(r"(\d+(?:,\s*\d+)*)\s*BHK",i.text)
        nums=re.findall(r'\d+', p.group(1))
        if p:
            for n in nums:
                p=re.findall("\s+\w+\w+?\s(.\w+,\s+\d+)\s?\w+",i.text)
                if p:
                    date.append(p[0])
                else:
                    date.append('NA')


In [30]:
a=[Names,types,price,Area,city,status,date,builder]

for i in a:
    print(len(i))

128
128
128
128
128
128
128
128


In [8]:
columns=["Name","Builder","Type","Price","Area","City","Status","Launch_Date"]

In [9]:
columns

['Name', 'Builder', 'Type', 'Price', 'Area', 'City', 'Status', 'Launch_Date']

In [31]:
df_3=pd.DataFrame({"Name":Names,"Builder":builder,"Type of Appartment(BHK)":types,"Price":price,"Area":Area,"City":city,"Status":status,"Date of Construction":date,})

In [32]:
df_3

,Name,Builder,Type of Appartment(BHK),Price,Area,City,Status,Date of Construction
0,Ultima Nest,Kamlesh Gandhi Projects,3,₹77 - 88 L,Waghodia Road,Vadodara,New Launch,"Apr, 2029"
1,Ultima Nest,Kamlesh Gandhi Projects,4,₹1.13 Cr,Waghodia Road,Vadodara,New Launch,"Apr, 2029"
2,Gajanana Park Street,Gajanana Properties,4,₹1.03 - 1.08 Cr,Tarsali,Vadodara,Under Construction,"Jun, 2026"
3,Hari Darshan Prime,Aarchi Developers,3,₹57 L,Chhani,Vadodara,Under Construction,"Dec, 2027"
4,DNG Noble Sky,DNG Group Vadodara,3,₹65 - 75 L,Gotri,Vadodara,New Launch,"Dec, 2029"
...,...,...,...,...,...,...,...,...
123,Dwaarika Heights,Prasandi Group,4,₹1.15 - 1.29 Cr,Ved Vyas Puri,Meerut,Ready To Move,"Mar, 2020"
124,Ashapurna NRI,Ashapurna Buildcon,2,₹38 L,Pali Road,Jodhpur,Ready To Move,"Oct, 2024"
125,Ashapurna NRI,Ashapurna Buildcon,3,₹81 L,Pali Road,Jodhpur,Ready To Move,"Oct, 2024"
126,Ashiana Dwarka,Ashiana Housing,2,₹50.29 L,Pal-Sangriya Link Road,Jodhpur,Under Construction,"Jul, 2026"


In [ ]:
df1["City"]

In [84]:
df=pd.read_excel("Real Estate Project Dataframe_1.xlsx")

In [85]:
df2=pd.read_excel("Real_Estate_Project_Dataframe.xlsx")

In [86]:
df

,Name,Builder,Type of Appartment(BHK),Price,Area,City,Status,Date of Construction
0,Urban Vista,Loharuka Group,2,₹75 L - 1.02 Cr,Rajarhat,Kolkata,Under Construction,"Dec, 2026"
1,Urban Vista,Loharuka Group,3,₹1.02 - 1.17 Cr,Rajarhat,Kolkata,Under Construction,"Dec, 2026"
2,Oswal Orchard Amritaya,Oswal Group,3,₹1.14 - 1.26 Cr,BT Road,Kolkata,Under Construction,"Aug, 2027"
3,Loharuka Green Heights Phase,Loharuka Group,2,₹60.18 - 69.47 L,Rajarhat,Kolkata,Under Construction,"Jun, 2026"
4,Loharuka Green Heights Phase,Loharuka Group,3,₹77.01 - 88.15 L,Rajarhat,Kolkata,Under Construction,"Jun, 2026"
...,...,...,...,...,...,...,...,...
307,Skyblue Oasis,Pankaj Construction,2,₹46.04 - 47.68 L,Manish Nagar,Nagpur,Under Construction,"Dec, 2028"
308,Skyblue Oasis,Pankaj Construction,3,₹61.88 - 67.68 L,Manish Nagar,Nagpur,Under Construction,"Dec, 2028"
309,Skyblue Oasis,Pankaj Construction,4,₹70.6 - 81.96 L,Manish Nagar,Nagpur,Under Construction,"Dec, 2028"
310,Greenfield,LG Developers,2,₹26.99 - 33.49 L,Hingna Road,Nagpur,Under Construction,"Aug, 2026"


In [87]:
df1

,Name,Builder,Type of Appartment(BHK),Price,Area,City,Status,Date of Construction
0,Mundeshwari Megapolis,Mundeshwari Group,3,₹1.05 - 1.57 Cr,Danapur,Patna,New Launch,"Jul, 2031"
1,Winsome Icon,Winsome Realtors,2,₹1.2 - 1.43 Cr,Gola Road,Patna,Under Construction,"Mar, 2028"
2,Winsome Icon,Winsome Realtors,3,₹1.45 - 1.88 Cr,Gola Road,Patna,Under Construction,"Mar, 2028"
3,Winsome Icon,Winsome Realtors,4,₹1.95 - 2.27 Cr,Gola Road,Patna,Under Construction,"Mar, 2028"
4,Anshul H,Anshul Group,2,₹70 - 71.02 L,Shivala Par,Patna,Under Construction,"Jul, 2026"
...,...,...,...,...,...,...,...,...
219,Global Lake View,Global Space Developers,3,Price on Request,Six Mile,Guwahati,New Launch,"May, 2027"
220,Inclts Orchhid,Inclits Core Ventures,3,₹68 - 81.32 L,Bhetapara,Guwahati,Under Construction,"May, 2026"
221,Roodraksh Pride,Roodraksh,3,₹99.25 L,Zoo Tiniali,Guwahati,Ready To Move,"Oct, 2024"
222,Palacia Majestica,MP Construction Guwahati,2,Price on Request,Panjabari,Guwahati,New Launch,"Aug, 2031"


In [88]:
df2

,Name,Builder,Type of Appartment(BHK),Price,Area,City,Status,Date of Construction
0,Bricks Marvella,Bricks Ramabhupal Projects LLP,3,₹1.33 - 1.83 Cr,Tellapur,Hyderabad,Under Construction,"Dec, 2027"
1,Bricks Marvella,Bricks Ramabhupal Projects LLP,4,₹2.64 Cr,Tellapur,Hyderabad,Under Construction,"Dec, 2027"
2,URBANYARDS SHANGRILA,TempleTree Developers LLP,2,₹87.96 L,Tukkuguda,Hyderabad,New Launch,"Jun, 2028"
3,URBANYARDS SHANGRILA,TempleTree Developers LLP,3,₹1.06 - 1.74 Cr,Tukkuguda,Hyderabad,New Launch,"Jun, 2028"
4,GHR Trivana,GHR INFRA,4,₹3.66 Cr,Tukkuguda,Hyderabad,New Launch,"Feb, 2030"
...,...,...,...,...,...,...,...,...
400,Casadel Above The World,Casadel,4,₹1.37 - 1.59 Cr,Kakkanad,Kochi,Under Construction,"Mar, 2026"
401,Vishraam Vismaya,Vishraam,2,₹74 L,Manakkakadav,Kochi,New Launch,"Oct, 2030"
402,Vishraam Vismaya,Vishraam,3,₹1.04 Cr,Manakkakadav,Kochi,New Launch,"Oct, 2030"
403,Confident Lakeside,Confident Group,2,₹70 L,Kakkanad,Kochi,New Launch,"Jan, 2031"


In [34]:
DF_2=pd.concat([DF_1,df_3], axis=0) 

In [36]:
DF_2.to_excel("Full_DataFrame_Real_Estate_2.xlsx",index=False)

In [23]:
DF=pd.read_excel("Full_DataFrame_Real_Estate.xlsx")

In [35]:
DF_2

,Name,Builder,Type of Appartment(BHK),Price,Area,City,Status,Date of Construction
0,Bricks Marvella,Bricks Ramabhupal Projects LLP,3,₹1.33 - 1.83 Cr,Tellapur,Hyderabad,Under Construction,"Dec, 2027"
1,Bricks Marvella,Bricks Ramabhupal Projects LLP,4,₹2.64 Cr,Tellapur,Hyderabad,Under Construction,"Dec, 2027"
2,URBANYARDS SHANGRILA,TempleTree Developers LLP,2,₹87.96 L,Tukkuguda,Hyderabad,New Launch,"Jun, 2028"
3,URBANYARDS SHANGRILA,TempleTree Developers LLP,3,₹1.06 - 1.74 Cr,Tukkuguda,Hyderabad,New Launch,"Jun, 2028"
4,GHR Trivana,GHR INFRA,4,₹3.66 Cr,Tukkuguda,Hyderabad,New Launch,"Feb, 2030"
...,...,...,...,...,...,...,...,...
123,Dwaarika Heights,Prasandi Group,4,₹1.15 - 1.29 Cr,Ved Vyas Puri,Meerut,Ready To Move,"Mar, 2020"
124,Ashapurna NRI,Ashapurna Buildcon,2,₹38 L,Pali Road,Jodhpur,Ready To Move,"Oct, 2024"
125,Ashapurna NRI,Ashapurna Buildcon,3,₹81 L,Pali Road,Jodhpur,Ready To Move,"Oct, 2024"
126,Ashiana Dwarka,Ashiana Housing,2,₹50.29 L,Pal-Sangriya Link Road,Jodhpur,Under Construction,"Jul, 2026"


In [7]:
import requests

url = "https://www.99acres.com"

headers = {"User-Agent":"Mozilla/5.0"}

print(requests.get(url, headers=headers).status_code)

403
